#  Импорты и базовые настройки

In [1]:
# ==============================================================================
# ЯЧЕЙКА 1: Импорты, Настройки и Устройство
# ==============================================================================
import os
import re
import time
import random
import gc # Сборщик мусора для очистки памяти
import numpy as np
import cv2
import tifffile as tiff
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Устройство: {device}")
if torch.cuda.is_available():
    print(f"✅ Видеокарта: {torch.cuda.get_device_name(0)}")
    print(f"✅ Доступно VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GLOBAL_MAX = 65535.0

✅ Устройство: cuda
✅ Видеокарта: Tesla V100-PCIE-32GB
✅ Доступно VRAM: 34.08 GB


# Архитектура (Guided Restormer + Noise Estimator)

In [2]:
# ==============================================================================
# ЯЧЕЙКА 2: Архитектура (Guided Restormer)
# ==============================================================================
class SigmaEstimator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, 16), nn.ReLU(inplace=True),
            nn.Linear(16, 1), nn.Softplus()
        )
    def forward(self, x): return self.net(x)

class MDTA(nn.Module):
    def __init__(self, channels, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(1, num_heads, 1, 1))
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1, bias=False)
        self.qkv_dwconv = nn.Conv2d(channels * 3, channels * 3, kernel_size=3, padding=1, groups=channels * 3, bias=False)
        self.project_out = nn.Conv2d(channels, channels, kernel_size=1, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv_dwconv(self.qkv(x))
        q, k, v = qkv.chunk(3, dim=1)
        q = q.reshape(b, self.num_heads, -1, h * w)
        k = k.reshape(b, self.num_heads, -1, h * w)
        v = v.reshape(b, self.num_heads, -1, h * w)
        q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
        attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * self.temperature, dim=-1)
        return self.project_out(torch.matmul(attn, v).reshape(b, c, h, w))

class GDFN(nn.Module):
    def __init__(self, channels, expansion_factor=2.66):
        super().__init__()
        hidden_channels = int(channels * expansion_factor)
        self.project_in = nn.Conv2d(channels, hidden_channels * 2, kernel_size=1, bias=False)
        self.dwconv = nn.Conv2d(hidden_channels * 2, hidden_channels * 2, kernel_size=3, padding=1, groups=hidden_channels * 2, bias=False)
        self.project_out = nn.Conv2d(hidden_channels, channels, kernel_size=1, bias=False)

    def forward(self, x):
        x1, x2 = self.dwconv(self.project_in(x)).chunk(2, dim=1)
        return self.project_out(F.gelu(x1) * x2)

class TransformerBlock(nn.Module):
    def __init__(self, channels, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.attn = MDTA(channels, num_heads)
        self.norm2 = nn.LayerNorm(channels)
        self.ffn = GDFN(channels)

    def forward(self, x):
        x_norm = self.norm1(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        x = x + self.attn(x_norm)
        x_norm = self.norm2(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        x = x + self.ffn(x_norm)
        return x

class GuidedRestormer(nn.Module):
    def __init__(self, dim=48, num_blocks=4, num_heads=4):
        super().__init__()
        self.sigma_estimator = SigmaEstimator()
        self.embed = nn.Conv2d(3, dim, kernel_size=3, padding=1)
        self.blocks = nn.Sequential(*[TransformerBlock(dim, num_heads) for _ in range(num_blocks)])
        self.mapping = nn.Conv2d(dim, 1, kernel_size=3, padding=1)

    def forward(self, noisy, canny, true_sigma=None):
        pred_sigma = self.sigma_estimator(noisy)
        sigma_val = true_sigma if (self.training and true_sigma is not None) else pred_sigma
        B, _, H, W = noisy.shape
        sigma_map = sigma_val.view(B, 1, 1, 1).expand(B, 1, H, W)
        
        x = torch.cat([noisy, canny, sigma_map], dim=1)
        fea = self.embed(x)
        fea = self.blocks(fea)
        predicted_noise = self.mapping(fea)
        
        cleaned = noisy - predicted_noise
        return torch.clamp(cleaned, 0.0, 1.0), pred_sigma

# Оптимизированный Data Pipeline 

In [3]:
import cv2
import random
import numpy as np
import tifffile as tiff
import torch
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

def get_canny_full_image(img_numpy):
    """Применяет Canny к полной картинке."""
    img_8u = np.clip((img_numpy / GLOBAL_MAX) * 255, 0, 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(img_8u, (5, 5), 0)
    edges = cv2.Canny(blurred, 65, 140)
    return edges

def load_chunk_data(chunk_pairs, patch_size=512, overlap=0.2):
    """Загружает только 20 файлов (Чанк) в RAM."""
    stride = int(patch_size * (1.0 - overlap))
    noisy_imgs, clean_imgs, canny_imgs = [], [], []
    patch_coords = []
    
    for img_idx, (noisy_path, clean_path) in enumerate(chunk_pairs):
        n_img = tiff.imread(noisy_path)
        c_img = tiff.imread(clean_path)
        canny_img = get_canny_full_image(n_img)
        
        noisy_imgs.append(n_img)
        clean_imgs.append(c_img)
        canny_imgs.append(canny_img)
        
        h, w = n_img.shape
        for y in range(0, h - patch_size + 1, stride):
            for x in range(0, w - patch_size + 1, stride):
                patch_coords.append((img_idx, y, x))
                
    return noisy_imgs, clean_imgs, canny_imgs, patch_coords

def d4_augmentation_no_crop(noisy_t, clean_t, canny_t):
    """Аугментация СТРОГО для 3 тензоров. Обязательно возвращает 3 тензора."""
    if random.random() > 0.5:
        noisy_t = TF.hflip(noisy_t)
        clean_t = TF.hflip(clean_t)
        canny_t = TF.hflip(canny_t)
    if random.random() > 0.5:
        noisy_t = TF.vflip(noisy_t)
        clean_t = TF.vflip(clean_t)
        canny_t = TF.vflip(canny_t)
        
    k = random.randint(0, 3)
    if k > 0:
        noisy_t = torch.rot90(noisy_t, k, dims=[1, 2])
        clean_t = torch.rot90(clean_t, k, dims=[1, 2])
        canny_t = torch.rot90(canny_t, k, dims=[1, 2])
        
    return noisy_t, clean_t, canny_t

class ChunkDataset(Dataset):
    def __init__(self, noisy_imgs, clean_imgs, canny_imgs, coords, patch_size=512, apply_aug=True):
        self.noisy_imgs = noisy_imgs
        self.clean_imgs = clean_imgs
        self.canny_imgs = canny_imgs
        self.coords = coords
        self.patch_size = patch_size
        self.apply_aug = apply_aug

    def __len__(self): return len(self.coords)

    def __getitem__(self, idx):
        img_idx, y, x = self.coords[idx]
        p = self.patch_size
        
        n_patch = self.noisy_imgs[img_idx][y:y+p, x:x+p]
        c_patch = self.clean_imgs[img_idx][y:y+p, x:x+p]
        canny_patch = self.canny_imgs[img_idx][y:y+p, x:x+p]
        
        n_patch = np.clip(n_patch.astype(np.float32) / GLOBAL_MAX, 0.0, 1.0)
        c_patch = np.clip(c_patch.astype(np.float32) / GLOBAL_MAX, 0.0, 1.0)
        canny_patch = canny_patch.astype(np.float32) / 255.0
        
        sigma_val = float(np.std(n_patch - c_patch))
        noise_map = np.full((p, p), sigma_val, dtype=np.float32)
        
        n_combined = np.stack([n_patch, canny_patch, noise_map], axis=0) 
        n_t = torch.from_numpy(n_combined).float()
        c_t = torch.from_numpy(c_patch).unsqueeze(0).float()
        sigma_t = torch.tensor(sigma_val, dtype=torch.float32)
        
        if self.apply_aug:
            n_img_t = n_t[0:1, :, :]
            can_img_t = n_t[1:2, :, :]
            
            # Применяем аугментацию
            n_img_t, c_t, can_img_t = d4_augmentation_no_crop(n_img_t, c_t, can_img_t)
            
            # Защита от ошибок копирования: проверяем, что аугментация ничего не сломала
            assert n_img_t is not None, "Ошибка: Аугментация вернула None для картинки!"
            
            n_t[0:1, :, :] = n_img_t
            n_t[1:2, :, :] = can_img_t
            
        return n_t, c_t, sigma_t

print("✅ Пайплайн данных успешно загружен!")

✅ Пайплайн данных успешно загружен!


# Инициализация функций потерь (Loss Functions)

In [4]:
# ==============================================================================
# ЯЧЕЙКА 4: Функции потерь
# ==============================================================================
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps2 = eps ** 2
    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target)**2 + self.eps2))

class EdgeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def forward(self, pred, target):
        pred_x = F.conv2d(pred, self.sobel_x.to(pred.device), padding=1)
        pred_y = F.conv2d(pred, self.sobel_y.to(pred.device), padding=1)
        target_x = F.conv2d(target, self.sobel_x.to(pred.device), padding=1)
        target_y = F.conv2d(target, self.sobel_y.to(pred.device), padding=1)
        return F.l1_loss(pred_x, target_x) + F.l1_loss(pred_y, target_y)

class FullHybridCTLoss(nn.Module):
    def __init__(self, w_charb=0.5, w_edge=0.5):
        super().__init__()
        self.w_charb = w_charb
        self.w_edge = w_edge
        self.charb = CharbonnierLoss()
        self.edge = EdgeLoss()

    def forward(self, pred, target):
        return (self.w_charb * self.charb(pred, target)) + (self.w_edge * self.edge(pred, target))

# Главный цикл обучения

In [7]:
# ==============================================================================
# ПОДГОТОВКА СПИСКОВ ФАЙЛОВ (Train / Val / Test)
# ==============================================================================
import os
import re
import random

# УБЕДИТЕСЬ, ЧТО ПУТИ ПРАВИЛЬНЫЕ!
NOISY_DIR = "filestore/filestorage/V_beton30_angle05"
CLEAN_DIR = "filestore/filestorage/V_beton30_angle005"

def get_file_number(filename):
    nums = re.findall(r'\d+', filename)
    return int(nums[-1]) if nums else -1

# 1. Поиск всех файлов
noisy_files = sorted([f for f in os.listdir(NOISY_DIR) if f.lower().endswith(('.tif', '.tiff'))])
clean_files = sorted([f for f in os.listdir(CLEAN_DIR) if f.lower().endswith(('.tif', '.tiff'))])
all_common_files = sorted(list(set(noisy_files) & set(clean_files)))

# 2. Фильтрация: оставляем только полезный диапазон (400 - 2430)
valid_files = [f for f in all_common_files if 400 <= get_file_number(f) <= 2430]

# 3. Формируем пары полных путей: (путь_к_шумному, путь_к_чистому)
all_pairs = [(os.path.join(NOISY_DIR, f), os.path.join(CLEAN_DIR, f)) for f in valid_files]

# 4. Перемешиваем с фиксированным seed для воспроизводимости
random.seed(42)
random.shuffle(all_pairs)

# 5. Разбиваем: 80% Train, 10% Val, 10% Test
n_total = len(all_pairs)
n_train = int(0.80 * n_total)
n_val = int(0.10 * n_total)

train_files_list = all_pairs[:n_train]
val_files_list = all_pairs[n_train:n_train + n_val]
test_files_list = all_pairs[n_train + n_val:]

print(f"📊 СТАТИСТИКА ДАТАСЕТА:")
print(f" 🟢 Обучение (Train): {len(train_files_list)} пар снимков")
print(f" 🟡 Валидация (Val):  {len(val_files_list)} пар снимков")
print(f" 🔴 Тест (Test):      {len(test_files_list)} пар снимков")

📊 СТАТИСТИКА ДАТАСЕТА:
 🟢 Обучение (Train): 1624 пар снимков
 🟡 Валидация (Val):  203 пар снимков
 🔴 Тест (Test):      204 пар снимков


In [ ]:
# ==============================================================================
# ЯЧЕЙКА 5: Цикл обучения (Чанки + Gradient Accumulation)
# ==============================================================================

NUM_EPOCHS = 50
IMAGES_PER_CHUNK = 20    # 🔥 ГЛОБАЛЬНАЯ КОНСТАНТА: Сколько файлов грузить за раз
BATCH_SIZE = 4           # 🔥 МАЛЕНЬКИЙ БАТЧ: Чтобы поместиться в 32GB VRAM
ACCUMULATION_STEPS = 4   # 🔥 НАКОПЛЕНИЕ: Эффективный батч = 4 * 4 = 16

model = GuidedRestormer().to(device)
criterion_img = FullHybridCTLoss(w_charb=0.5, w_edge=0.5) 
criterion_sigma = nn.L1Loss() 

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

print(f"🚀 ЗАПУСК ОБУЧЕНИЯ (Чанки: {IMAGES_PER_CHUNK} файлов | Батч: {BATCH_SIZE} | Накопление: {ACCUMULATION_STEPS})")

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*50}\nЭПОХА {epoch+1}/{NUM_EPOCHS}\n{'='*50}")
    
    # Перемешиваем файлы перед каждой эпохой
    random.shuffle(train_files_list)
    
    running_loss_img = 0.0
    total_batches = 0
    
    # Разбиваем весь датасет на ЧАНКИ
    for chunk_start in range(0, len(train_files_list), IMAGES_PER_CHUNK):
        chunk_pairs = train_files_list[chunk_start : chunk_start + IMAGES_PER_CHUNK]
        chunk_idx = (chunk_start // IMAGES_PER_CHUNK) + 1
        total_chunks = (len(train_files_list) + IMAGES_PER_CHUNK - 1) // IMAGES_PER_CHUNK
        
        # 1. Загружаем ЧАНК в RAM
        print(f"\n📦 Чанк {chunk_idx}/{total_chunks} (Файлы {chunk_start} - {chunk_start+len(chunk_pairs)})")
        noisy_imgs, clean_imgs, canny_imgs, coords = load_chunk_data(chunk_pairs, patch_size=512)
        
        # 2. Создаем DataLoader для ЧАНКА
        chunk_dataset = ChunkDataset(noisy_imgs, clean_imgs, canny_imgs, coords, apply_aug=True)
        chunk_loader = DataLoader(chunk_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        
        model.train()
        optimizer.zero_grad(set_to_none=True) # Обнуляем перед началом чанка
        
        pbar = tqdm(chunk_loader, desc=f"Обучение чанка {chunk_idx}")
        
        for batch_idx, (n_b, c_b, sigma_b) in enumerate(pbar):
            noisy_b = n_b[:, 0:1, :, :].to(device, non_blocking=True)
            canny_b = n_b[:, 1:2, :, :].to(device, non_blocking=True)
            n_b = n_b.to(device, non_blocking=True) 
            c_b = c_b.to(device, non_blocking=True)
            sigma_b = sigma_b.to(device, non_blocking=True).unsqueeze(1) 
            
            with torch.cuda.amp.autocast():
                pred_clean, pred_sigma = model(noisy_b, canny_b, true_sigma=sigma_b)
                loss_img = criterion_img(pred_clean, c_b)         
                loss_sigma = criterion_sigma(pred_sigma, sigma_b) 
                
                # Делим лосс на кол-во шагов накопления!
                total_loss = (loss_img + 0.1 * loss_sigma) / ACCUMULATION_STEPS
                
            scaler.scale(total_loss).backward()
            
            # Шаг оптимизатора только каждые ACCUMULATION_STEPS батчей
            if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(chunk_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            
            running_loss_img += loss_img.item()
            total_batches += 1
            pbar.set_postfix(L_img=f"{loss_img.item():.4f}")
            
        # 3. ОЧИСТКА ПАМЯТИ ПОСЛЕ ЧАНКА (Жестко удаляем всё)
        del noisy_imgs, clean_imgs, canny_imgs, coords, chunk_dataset, chunk_loader
        gc.collect()
        torch.cuda.empty_cache()
        
    avg_img_loss = running_loss_img / total_batches
    print(f"🏁 Эпоха {epoch+1} завершена. Avg Hybrid Loss: {avg_img_loss:.4f}")
    
    torch.save(model.state_dict(), f"restormer_guided_epoch_{epoch+1}.pth")

🚀 ЗАПУСК ОБУЧЕНИЯ (Чанки: 20 файлов | Батч: 4 | Накопление: 4)

ЭПОХА 1/50

📦 Чанк 1/82 (Файлы 0 - 20)


Обучение чанка 1:  93%|█████████▎| 228/245 [02:30<00:10,  1.55it/s, L_img=0.2706]

In [1]:
# ==============================================================================
# 1. ИМПОРТ БИБЛИОТЕК И НАСТРОЙКА DATASPHERE
# ==============================================================================
import os
# Безопасная настройка под PyTorch 2.0.1 (без крашей ядра)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import gc
import re
import csv
import json
import time
import random
import numpy as np
import cv2
import tifffile as tiff
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

# Очистка памяти перед стартом
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ DataSphere GPU активен: {gpu_name} (Всего VRAM: {vram_gb:.1f} ГБ)")
else:
    device = torch.device("cpu")
    print("⚙️ Вычисления на CPU")

print("✅ Все библиотеки успешно импортированы!")

⚙️ Вычисления на CPU
✅ Все библиотеки успешно импортированы!


In [2]:
# ==============================================================================
# МОДУЛЬ 1: Чистая функция нарезки (Slicer)
# ==============================================================================
def slice_image_to_patches(image_or_path, patch_size=512, overlap_ratio=0.2):
    """
    ОТВЕТСТВЕННОСТЬ: Только геометрическая нарезка снимка.
    Возвращает:
      - patches: numpy массив (N, patch_size, patch_size)
      - coords: список координат (y, x)
      - orig_shape: размеры (H, W)
    """
    if isinstance(image_or_path, str):
        img = tiff.imread(image_or_path).astype(np.float32)
    else:
        img = image_or_path.astype(np.float32)

    h, w = img.shape
    stride = int(patch_size * (1.0 - overlap_ratio))

    y_steps = list(range(0, h - patch_size + 1, stride))
    if y_steps[-1] != h - patch_size: y_steps.append(h - patch_size)

    x_steps = list(range(0, w - patch_size + 1, stride))
    if x_steps[-1] != w - patch_size: x_steps.append(w - patch_size)

    patches, coords = [], []
    for y in y_steps:
        for x in x_steps:
            patches.append(img[y:y+patch_size, x:x+patch_size])
            coords.append((y, x))

    return np.array(patches), coords, (h, w)

print("✅ Модуль 1 (Нарезчик Slicer) готов!")

✅ Модуль 1 (Нарезчик Slicer) готов!


In [3]:
# ==============================================================================
# МОДУЛЬ 2: Аугментация D4 + Параметр Гамма-коррекции для (Noisy, Clean, Canny)
# ==============================================================================
def d4_gamma_canny_augmentation(noisy_tensor, clean_tensor, canny_tensor, gamma_range=(0.9, 1.1)):
    """
    Синхронная аугментация для 3-х карт:
    1. Гамма-коррекция (случайное изменение контраста I^gamma)
    2. Отражения по горизонтали и вертикали (Flips)
    3. Повороты на 90, 180, 270 градусов
    """
    # 1. 🔥 ПАРАМЕТР ГАММА
    if gamma_range is not None:
        gamma = random.uniform(gamma_range[0], gamma_range[1])
        noisy_tensor = torch.clamp(noisy_tensor ** gamma, 0.0, 1.0)
        clean_tensor = torch.clamp(clean_tensor ** gamma, 0.0, 1.0)

    # 2. Отражения (50% шанс)
    if random.random() > 0.5:
        noisy_tensor = TF.hflip(noisy_tensor)
        clean_tensor = TF.hflip(clean_tensor)
        canny_tensor = TF.hflip(canny_tensor)
        
    if random.random() > 0.5:
        noisy_tensor = TF.vflip(noisy_tensor)
        clean_tensor = TF.vflip(clean_tensor)
        canny_tensor = TF.vflip(canny_tensor)

    # 3. Повороты кратно 90°
    k = random.randint(0, 3)
    if k > 0:
        noisy_tensor = torch.rot90(noisy_tensor, k, dims=[1, 2])
        clean_tensor = torch.rot90(clean_tensor, k, dims=[1, 2])
        canny_tensor = torch.rot90(canny_tensor, k, dims=[1, 2])

    return noisy_tensor, clean_tensor, canny_tensor

print("✅ Модуль 2 (Аугментация с Гаммой и Canny) готов!")

✅ Модуль 2 (Аугментация с Гаммой и Canny) готов!


In [4]:
# ==============================================================================
# МОДУЛЬ 3: Ультра-легкий буферный датасет (Генерирует Noisy + Canny на лету)
# ==============================================================================
class CTBufferedDataset(Dataset):
    """
    Буферный датасет:
    - Считывает снимок 3067x3067 ровно 1 раз
    - Генерирует карту Canny в компактном uint8
    - Формирует 2-канальный вход [Noisy, Canny]
    - Расход памяти: всего ~150-200 МБ
    """
    def __init__(
        self, 
        noisy_dir, 
        clean_dir, 
        file_list=None, 
        patch_size=512, 
        overlap_ratio=0.2, 
        chunk_size=8,
        transform_fn=None
    ):
        self.patch_size = patch_size
        self.chunk_size = chunk_size
        self.transform_fn = transform_fn

        # 1. Список файлов
        if file_list is not None:
            self.common_names = file_list
        else:
            noisy_files = sorted([f for f in os.listdir(noisy_dir) if f.lower().endswith(('.tif', '.tiff'))])
            clean_files = sorted([f for f in os.listdir(clean_dir) if f.lower().endswith(('.tif', '.tiff'))])
            self.common_names = sorted(list(set(noisy_files) & set(clean_files)))

        self.noisy_paths = [os.path.join(noisy_dir, f) for f in self.common_names]
        self.clean_paths = [os.path.join(clean_dir, f) for f in self.common_names]

        # 2. Базовые координаты сетки
        _, self.base_coords, _ = slice_image_to_patches(self.noisy_paths[0], patch_size, overlap_ratio)

        self._buffer_cache = {}
        self.patch_index = []
        self.reshuffle()

        print(f"📁 Загружено пар снимков: {len(self.common_names)} (от {self.common_names[0]} до {self.common_names[-1]})")
        print(f"🎉 Сформирован буферный индекс: {len(self.patch_index)} патчей (Буфер = {chunk_size} снимков ~ 150 МБ RAM)")
        print(f"🔄 Стратегия аугментации: {transform_fn.__name__ if transform_fn else 'Без аугментации'}\n")

    def reshuffle(self):
        """Перемешивает снимки и жестко очищает кэш перед новой эпохой."""
        self._buffer_cache.clear()
        gc.collect()

        self.patch_index = []
        img_indices = list(range(len(self.common_names)))
        random.shuffle(img_indices)

        for i in range(0, len(img_indices), self.chunk_size):
            chunk_imgs = img_indices[i:i + self.chunk_size]
            chunk_patches = []
            for img_idx in chunk_imgs:
                for y, x in self.base_coords:
                    chunk_patches.append((img_idx, y, x))
            
            random.shuffle(chunk_patches)
            self.patch_index.extend(chunk_patches)

    def __len__(self):
        return len(self.patch_index)

    def _load_image_pair_to_buffer(self, img_idx):
        """Считывает снимок 1 раз и генерирует Canny в uint8."""
        n_raw = tiff.imread(self.noisy_paths[img_idx]).astype(np.float32)
        c_raw = tiff.imread(self.clean_paths[img_idx]).astype(np.float32)

        # Нормализация в компактный uint8 [0..255] (экономит 75% RAM)
        n_u8 = ((n_raw - np.min(n_raw)) / (np.max(n_raw) - np.min(n_raw) + 1e-8) * 255.0).astype(np.uint8)
        c_u8 = ((c_raw - np.min(c_raw)) / (np.max(c_raw) - np.min(c_raw) + 1e-8) * 255.0).astype(np.uint8)

        # 🔥 Генерируем карту Canny
        blurred = cv2.GaussianBlur(n_u8, (3, 3), 1.0)
        canny_u8 = cv2.Canny(blurred, 65, 140)

        self._buffer_cache[img_idx] = (n_u8, c_u8, canny_u8)

    def __getitem__(self, idx):
        img_idx, y, x = self.patch_index[idx]

        if img_idx not in self._buffer_cache:
            if len(self._buffer_cache) >= self.chunk_size:
                oldest_key = next(iter(self._buffer_cache))
                del self._buffer_cache[oldest_key]

            self._load_image_pair_to_buffer(img_idx)

        n_u8, c_u8, canny_u8 = self._buffer_cache[img_idx]
        p = self.patch_size

        n_patch = n_u8[y:y+p, x:x+p]
        c_patch = c_u8[y:y+p, x:x+p]
        canny_patch = canny_u8[y:y+p, x:x+p]

        # Перевод в тензоры [1, H, W]
        n_tensor = torch.from_numpy(n_patch).unsqueeze(0).float() / 255.0
        c_tensor = torch.from_numpy(c_patch).unsqueeze(0).float() / 255.0
        canny_tensor = torch.from_numpy(canny_patch).unsqueeze(0).float() / 255.0

        # Применяем аугментацию (D4 + Гамма)
        if self.transform_fn is not None:
            n_tensor, c_tensor, canny_tensor = self.transform_fn(n_tensor, c_tensor, canny_tensor)

        # 🔥 Склеиваем Noisy (1 канал) + Canny (1 канал) в 2-канальный вход [2, 512, 512]
        input_2ch = torch.cat([n_tensor, canny_tensor], dim=0)

        return input_2ch, c_tensor

print("✅ Модуль 3 (Датасет с 2-канальным входом Noisy + Canny) готов!")

✅ Модуль 3 (Датасет с 2-канальным входом Noisy + Canny) готов!


In [5]:
# ==============================================================================
# ⚙️ СТАБИЛЬНЫЕ ПАРАМЕТРЫ ДЛЯ RESTORMER (VRAM: ~6.5 ГБ из 32 ГБ)
# ==============================================================================
PATCH_SIZE = 256 # 🔥 256x256 снижает расход VRAM в 4 раза!
CHUNK_SIZE = 8   # 8 снимков в буфере RAM (~150 МБ)
STEP = 5         # Каждый 5-й снимок
BATCH_SIZE = 4   # 🔥 Батч 4 гарантирует 100% стабильность без OOM

NOISY_DIR = "filestore/filestorage/V_beton30_angle05"
CLEAN_DIR = "filestore/filestorage/V_beton30_angle005"
# ==============================================================================

def get_file_number(filename):
    nums = re.findall(r'\d+', filename)
    return int(nums[-1]) if nums else -1

# 1. Поиск всех пар файлов
noisy_files = sorted([f for f in os.listdir(NOISY_DIR) if f.lower().endswith(('.tif', '.tiff'))])
clean_files = sorted([f for f in os.listdir(CLEAN_DIR) if f.lower().endswith(('.tif', '.tiff'))])
all_pairs = sorted(list(set(noisy_files) & set(clean_files)))

# 2. Фильтрация краевых срезов [0..400] и [2430..2816] + шаг STEP=5
clean_range_pairs = [
    f for f in all_pairs 
    if not (0 <= get_file_number(f) <= 400 or 2430 <= get_file_number(f) <= 2816)
]
subsampled_pairs = clean_range_pairs[::STEP]

# 3. Воспроизводимое разделение 80 / 10 / 10 (seed=42)
random.seed(42)
shuffled_pairs = subsampled_pairs.copy()
random.shuffle(shuffled_pairs)

n_total = len(shuffled_pairs)          # 406 снимков
n_train = int(0.80 * n_total)          # 324 снимка (80%)
n_val   = int(0.10 * n_total)          # 40 снимков (10%)
n_test  = n_total - n_train - n_val    # 42 снимка (10%)

train_files = sorted(shuffled_pairs[:n_train])
val_files   = sorted(shuffled_pairs[n_train:n_train + n_val])
test_files  = sorted(shuffled_pairs[n_train + n_val:])

print("=" * 75)
print(f"📊 ПАРАМЕТРЫ RESTORMER (Патчи {PATCH_SIZE}x{PATCH_SIZE}, Батч {BATCH_SIZE}):")
print(f"   🟢 Train (80%): {len(train_files)} пар — D4 + Гамма")
print(f"   🟡 Val   (10%): {len(val_files)} пар — для чекпоинтов")
print(f"   🔴 Test  (10%): {len(test_files)} пар — НЕПРИКОСНОВЕННЫЙ ТЕСТ")
print("=" * 75 + "\n")

# 4. Создаем датасеты на 256x256
train_dataset = CTBufferedDataset(
    noisy_dir=NOISY_DIR,
    clean_dir=CLEAN_DIR,
    file_list=train_files,
    patch_size=PATCH_SIZE,                # 🔥 256x256
    overlap_ratio=0.2,
    chunk_size=CHUNK_SIZE,
    transform_fn=d4_gamma_canny_augmentation
)

val_dataset = CTBufferedDataset(
    noisy_dir=NOISY_DIR,
    clean_dir=CLEAN_DIR,
    file_list=val_files,
    patch_size=PATCH_SIZE,                # 🔥 256x256
    overlap_ratio=0.2,
    chunk_size=CHUNK_SIZE,
    transform_fn=None
)

test_dataset = CTBufferedDataset(
    noisy_dir=NOISY_DIR,
    clean_dir=CLEAN_DIR,
    file_list=test_files,
    patch_size=PATCH_SIZE,                # 🔥 256x256
    overlap_ratio=0.2,
    chunk_size=CHUNK_SIZE,
    transform_fn=None
)

# 5. DataLoader'ы
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"🎉 ИТОГ: Train={len(train_dataset)} | Val={len(val_dataset)} | Test={len(test_dataset)} (VRAM защищена!)")

📊 ПАРАМЕТРЫ RESTORMER (Патчи 256x256, Батч 4):
   🟢 Train (80%): 324 пар — D4 + Гамма
   🟡 Val   (10%): 40 пар — для чекпоинтов
   🔴 Test  (10%): 42 пар — НЕПРИКОСНОВЕННЫЙ ТЕСТ

📁 Загружено пар снимков: 324 (от rec_00401.tif до rec_02426.tif)
🎉 Сформирован буферный индекс: 72900 патчей (Буфер = 8 снимков ~ 150 МБ RAM)
🔄 Стратегия аугментации: d4_gamma_canny_augmentation

📁 Загружено пар снимков: 40 (от rec_00511.tif до rec_02386.tif)
🎉 Сформирован буферный индекс: 9000 патчей (Буфер = 8 снимков ~ 150 МБ RAM)
🔄 Стратегия аугментации: Без аугментации

📁 Загружено пар снимков: 42 (от rec_00416.tif до rec_02411.tif)
🎉 Сформирован буферный индекс: 9450 патчей (Буфер = 8 снимков ~ 150 МБ RAM)
🔄 Стратегия аугментации: Без аугментации

🎉 ИТОГ: Train=72900 | Val=9000 | Test=9450 (VRAM защищена!)


In [6]:
# =============================================================================
# КОД ДЛЯ ВЫВОДА ТЕСТОВОЙ ВЫБОРКИ (ВСТАВИТЬ В КАЖДЫЙ НОУТБУК)
# =============================================================================
import os
import json

# 1. Определяем переменную с тестовыми файлами (гибкий поиск)
# test_files = None

# Пробуем разные возможные имена переменных
if 'test_files' in globals():
    test_files = test_files
elif 'test_files_list' in globals():
    # В первом ноутбуке test_files_list это список пар (путь, путь)
    # Нам нужны только имена файлов
    if isinstance(test_files_list, list) and len(test_files_list) > 0:
        if isinstance(test_files_list[0], tuple) or isinstance(test_files_list[0], list):
            test_files = [os.path.basename(pair[0]) for pair in test_files_list]
        else:
            test_files = test_files_list
elif 'test_dataset' in globals():
    # Может быть переменная с датасетом
    if hasattr(test_dataset, 'common_names'):
        test_files = test_dataset.common_names
    elif hasattr(test_dataset, 'file_list'):
        test_files = test_dataset.file_list

if test_files is None:
    raise NameError("Не удалось найти тестовую выборку. Проверьте имя переменной.")

# 2. Выводим информацию
print("=" * 60)
print(f"📊 ТЕСТОВАЯ ВЫБОРКА: {len(test_files)} файлов")
print("=" * 60)
for i, fname in enumerate(test_files):
    print(f"{i+1:4d}. {fname}")

# 3. Сохраняем список в файл (для последующего сравнения)
output_file = "test_files_list.txt"
with open(output_file, "w") as f:
    for fname in test_files:
        f.write(fname + "\n")
print(f"\n✅ Список сохранён в файл: {output_file}")

# Также сохраняем в JSON для удобства
json_file = "test_files_list.json"
with open(json_file, "w") as f:
    json.dump(test_files, f, indent=2)
print(f"✅ Список сохранён в JSON: {json_file}")

📊 ТЕСТОВАЯ ВЫБОРКА: 42 файлов
   1. rec_00416.tif
   2. rec_00461.tif
   3. rec_00466.tif
   4. rec_00476.tif
   5. rec_00481.tif
   6. rec_00621.tif
   7. rec_00636.tif
   8. rec_00661.tif
   9. rec_00686.tif
  10. rec_00756.tif
  11. rec_00796.tif
  12. rec_00806.tif
  13. rec_00906.tif
  14. rec_00956.tif
  15. rec_00961.tif
  16. rec_00971.tif
  17. rec_00996.tif
  18. rec_01026.tif
  19. rec_01101.tif
  20. rec_01111.tif
  21. rec_01271.tif
  22. rec_01471.tif
  23. rec_01481.tif
  24. rec_01546.tif
  25. rec_01691.tif
  26. rec_01796.tif
  27. rec_01836.tif
  28. rec_01906.tif
  29. rec_01911.tif
  30. rec_01941.tif
  31. rec_02036.tif
  32. rec_02061.tif
  33. rec_02131.tif
  34. rec_02186.tif
  35. rec_02196.tif
  36. rec_02231.tif
  37. rec_02256.tif
  38. rec_02286.tif
  39. rec_02296.tif
  40. rec_02351.tif
  41. rec_02366.tif
  42. rec_02411.tif

✅ Список сохранён в файл: test_files_list.txt
✅ Список сохранён в JSON: test_files_list.json
